# Experiment: Basketball Ball Detection Training

Objective:
- Train a stronger ball detector for the local CourtVision pipeline.
- Default to the YOLO26 family because it is the latest Ultralytics detector family supported by Roboflow and Ultralytics.
- Publish the final checkpoint to `Models/ball_detector_model.pt` so the app can load it directly.


## Model choice

This notebook is optimized for the current repo, not just for standalone benchmark scores.
- Default profile: `balanced`.
- Recommended family: `YOLO26`.
- Why not switch to Roboflow-hosted RF-DETR here: the backend currently consumes local Ultralytics `.pt` weights directly, so YOLO26 is the strongest drop-in upgrade.
- Override with env vars if you want a heavier `quality` run or a lighter `edge` run.


In [ ]:
# Install dependencies once per fresh notebook runtime
%pip install -q "roboflow>=1.2.13" "ultralytics>=8.4.14" pyyaml pillow


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from ultralytics import YOLO

repo_root = Path.cwd().resolve()
if repo_root.name == "Training_Notebooks":
    repo_root = repo_root.parent
if not (repo_root / "Training_Notebooks").exists():
    raise FileNotFoundError("Run this notebook from the repository root or the Training_Notebooks directory.")

training_dir = repo_root / "Training_Notebooks"
if str(training_dir) not in sys.path:
    sys.path.insert(0, str(training_dir))

from yolo_training_common import (
    build_dataset_summary,
    build_training_config,
    build_training_kwargs,
    copy_best_weights,
    download_dataset,
    inspect_dataset_images,
    normalize_data_yaml,
    write_training_summary,
)

config = build_training_config(
    task_key="ball",
    env_prefix="BALL",
    output_model_path="Models/ball_detector_model.pt",
    default_workspace="cricket-qnb5l",
    default_project="basketball-xil7x",
    default_version=1,
)
config


## Plan

- Download the chosen Roboflow dataset version.
- Normalize `data.yaml` so the notebook does not depend on folder-moving hacks.
- Inspect class labels and sample image resolution before training.
- Train the detector with a preset tuned for small-object recall without making the final detector unnecessarily heavy.
- Validate the best checkpoint and copy it into `Models/`.
- Write a small training metadata JSON file next to the exported model.


In [ ]:
dataset_root = download_dataset(config)
data_yaml_path = normalize_data_yaml(dataset_root)
image_stats = inspect_dataset_images(data_yaml_path)
dataset_summary = build_dataset_summary(config, data_yaml_path, image_stats)
dataset_summary


In [ ]:
model = YOLO(config["base_model"])
train_results = model.train(**build_training_kwargs(config, data_yaml_path))

save_dir = Path(train_results.save_dir)
best_weights_path = save_dir / "weights" / "best.pt"
save_dir


In [ ]:
trained_model = YOLO(best_weights_path)
val_metrics = trained_model.val(
    data=str(data_yaml_path),
    imgsz=config["image_size"],
    batch=config["batch_size"],
    split="val",
    device=config["device"],
)

published_model_path = copy_best_weights(save_dir, config["output_model_path"])
summary_path = write_training_summary(
    config=config,
    dataset_summary=dataset_summary,
    best_weights_path=best_weights_path,
    published_model_path=published_model_path,
    map50=float(val_metrics.box.map50),
    map50_95=float(val_metrics.box.map),
)

result = {
    "task_name": config["task_name"],
    "profile": config["profile"],
    "base_model": config["base_model"],
    "best_weights_path": str(best_weights_path),
    "published_model_path": str(published_model_path),
    "summary_path": str(summary_path),
    "map50": float(val_metrics.box.map50),
    "map50_95": float(val_metrics.box.map),
}
result


## Next steps

- If the dataset warnings mention low export resolution, regenerate the Roboflow dataset version at a larger size before retraining.
- If this run is too slow for the app, switch to `COURTVISION_BALL_PROFILE=edge`.
- If recall is still weak and local speed is acceptable, switch to `COURTVISION_BALL_PROFILE=quality`.
- Keep the generated `*.train.json` file with the exported model so you know which dataset version produced it.
